In [1]:
import ee
import geemap
from tgbs_rs.config.config import GEE_PROJECT

In [ ]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

In [3]:
from pathlib import Path

from tgbs_rs.config.config import (
    S2_SCALE,
    AOI_PATHS,
    SPATIAL_CHANGE_DRIVE_FOLDER,
    DEFAULT_CRS,
    OUTPUT_DIRS,
)

from tgbs_rs.config.config_vis import SITES_VIS_PARAMS, DELTA_VIS, SLOPE_VIS, Z_ANOMALY_VIS, PERCENTILE_ANOMALY_VIS, FIRE_VIS

from tgbs_rs.io import export_spatial_products_to_drive


from tgbs_rs.utils import (
    build_default_sites_featurecollection,
    load_ks_rehab_blocks_featurecollection,
    load_site_feature
)

from tgbs_rs.data.sensors.sentinel.sentinel_preprocessing import (
    get_s2_sr_collection,
)

from tgbs_rs.metrics.spatial_change import (
    get_focal_buffered_geometry,
    build_baseline_current_delta,
    build_annual_single_index_collection,
    build_trend_slope_image,
    build_reference_site_value_featurecollection,
    build_reference_scalar_stats,
    build_reference_percentile_from_scalars,
    build_reference_z_anomaly_from_scalars,
    build_fire_dnbr_image,
    split_old_new_restoration_blocks,
    summarize_image_by_block_group,
)

In [ ]:
Map = geemap.Map()
Map.height = '800px'

In [4]:
PRIORITY_INDICES = ["NBR", "NDMI", "NDVI", "NIRv", "SAVI"]

# General focal-site restoration windows for whole-site heatmaps
SPATIAL_BASELINE_START = ee.Date("2019-01-01")
SPATIAL_BASELINE_END   = ee.Date("2021-12-31")
SPATIAL_CURRENT_START  = ee.Date("2023-01-01")
SPATIAL_CURRENT_END    = ee.Date("2025-12-31")

# Fire-specific windows
FIRE_PRE_START  = ee.Date("2024-07-01")
FIRE_PRE_END    = ee.Date("2024-12-31")
FIRE_POST_START = ee.Date("2025-01-01")
FIRE_POST_END   = ee.Date("2025-06-30")

In [5]:
# Build Site Feature Collection
sites_fc = build_default_sites_featurecollection()


reference_fc = sites_fc.filter(
    ee.Filter.eq("site_category", "reference")
)

degraded_fc = sites_fc.filter(
    ee.Filter.eq("site_category", "degraded")
)

focal_fc = sites_fc.filter(
    ee.Filter.eq("site_category", "focal")
)

##### Buffer the focal and corridor geometries by 500m and merge into a single geometry

In [6]:
ks_rehab_geom, ks_rehab_buffered = get_focal_buffered_geometry(
    focal_path=AOI_PATHS["ks_rehab_blocks"],
    buffer_m=500,
)

In [7]:
corridor_geom, corridor_buffered = get_focal_buffered_geometry(
    focal_path=AOI_PATHS["bio_corridor"],
    buffer_m=500,
)

In [8]:
merged_focal_geom = ks_rehab_buffered.union(corridor_buffered)

##### Build Sentinel-2 collection for the buffered focal AOI

In [9]:
s2_col = get_s2_sr_collection(
    aoi=merged_focal_geom,
    start_date=SPATIAL_BASELINE_START,
    end_date=SPATIAL_CURRENT_END,
    apply_water_masking=True,
)

##### Build baseline/current/delta rasters for priority indices

In [16]:
spatial_products = {}

for index_name in PRIORITY_INDICES:
    baseline_img, current_img, delta_img = build_baseline_current_delta(
        collection=s2_col,
        index_name=index_name,
        baseline_start=SPATIAL_BASELINE_START,
        baseline_end=SPATIAL_BASELINE_END,
        current_start=SPATIAL_CURRENT_START,
        current_end=SPATIAL_CURRENT_END,
        clip_geom=merged_focal_geom,
    )

    spatial_products[index_name] = {
        "baseline_mean": baseline_img,
        "current_mean": current_img,
        "delta": delta_img,
    }

##### Build annual collections for slope rasters

In [11]:
annual_index_collections = {}

for index_name in PRIORITY_INDICES:
    annual_index_collections[index_name] = build_annual_single_index_collection(
        collection=s2_col,
        index_name=index_name,
        start_date=SPATIAL_BASELINE_START,
        end_date=SPATIAL_CURRENT_END,
        composite_stat="median",
        clip_geom=merged_focal_geom,
    )

##### Build per-pixel slope images

In [12]:
for index_name in PRIORITY_INDICES:
    slope_img = build_trend_slope_image(
        annual_collection=annual_index_collections[index_name],
        index_name=index_name,
        clip_geom=merged_focal_geom,
    )
    spatial_products[index_name]["slope"] = slope_img

##### Build reference-relative current composites

In [ ]:
reference_value_tables = {}
reference_scalar_stats = {}

for index_name in PRIORITY_INDICES:
    ref_value_fc = build_reference_site_value_featurecollection(
        collection=s2_col,
        reference_fc=reference_fc,
        index_name=index_name,
        start_date=SPATIAL_CURRENT_START,
        end_date=SPATIAL_CURRENT_END,
        summary_stat="mean",   # or "median" if you want site composites to be median
        reducer=ee.Reducer.mean(),
        scale=10,
    )

    reference_value_tables[index_name] = ref_value_fc
    reference_scalar_stats[index_name] = build_reference_scalar_stats(
        ref_value_fc
    )

##### Build z-score and percentile anomaly rasters

In [ ]:
for index_name in PRIORITY_INDICES:
    focal_current_img = spatial_products[index_name]["current_mean"]

    z_img = build_reference_z_anomaly_from_scalars(
        focal_current_img=focal_current_img,
        reference_mean=reference_scalar_stats[index_name]["mean"],
        reference_std=reference_scalar_stats[index_name]["std"],
        index_name=index_name,
        clip_geom=merged_focal_geom,
    )

    pct_img = build_reference_percentile_from_scalars(
        focal_current_img=focal_current_img,
        reference_value_fc=reference_value_tables[index_name],
        index_name=index_name,
        clip_geom=merged_focal_geom,
    )

    spatial_products[index_name]["z_anomaly"] = z_img
    spatial_products[index_name]["percentile_anomaly"] = pct_img

##### Fire-specific NBR / dNBR workflow

In [18]:
fire_pre_nbr, fire_post_nbr, fire_dnbr = build_fire_dnbr_image(
    collection=s2_col,
    pre_start=FIRE_PRE_START,
    pre_end=FIRE_PRE_END,
    post_start=FIRE_POST_START,
    post_end=FIRE_POST_END,
    clip_geom=merged_focal_geom,
)

spatial_products["NBR"]["fire_pre_nbr"] = fire_pre_nbr
spatial_products["NBR"]["fire_post_nbr"] = fire_post_nbr
spatial_products["NBR"]["dnbr"] = fire_dnbr

##### Restoration Block cohort split: old vs new restoration blocks

In [ ]:
ks_blocks_fc = load_ks_rehab_blocks_featurecollection(
    AOI_PATHS["ks_rehab_blocks"],
    year_field="year",
    drop_null_year=True,
    drop_null_geometry=True,
)

In [ ]:
old_blocks_fc, new_blocks_fc = split_old_new_restoration_blocks(
    focal_blocks_path=AOI_PATHS["ks_rehab_blocks"],
)

##### Zonal summaries by block cohort

In [ ]:
cohort_summary_rows = []

for index_name in PRIORITY_INDICES:
    delta_img = spatial_products[index_name]["delta"]

    old_stats = summarize_image_by_block_group(
        image=delta_img,
        blocks_fc=old_blocks_fc,
        group_name="old",
        index_name=index_name,
        reducer=ee.Reducer.mean(),
        scale=10,
    )

    new_stats = summarize_image_by_block_group(
        image=delta_img,
        blocks_fc=new_blocks_fc,
        group_name="new",
        index_name=index_name,
        reducer=ee.Reducer.mean(),
        scale=10,
    )

    cohort_summary_rows.extend([old_stats, new_stats])

In [ ]:
import pandas as pd

rows = []

for item in cohort_summary_rows:
    rows.append(
        {
            "group_name": item["group_name"],
            "index_name": item["index_name"],
            "band_name": item["band_name"].getInfo(),
            **item["stats"].getInfo(),
        }
    )

cohort_summary_df = pd.DataFrame(rows)


In [ ]:
value_cols = [
    "NBR_delta",
    "NDMI_delta",
    "NDVI_delta",
    "NIRv_delta",
    "SAVI_delta",
]

cohort_summary_clean = cohort_summary_df.copy()

cohort_summary_clean["value"] = cohort_summary_clean[value_cols].bfill(axis=1).iloc[:, 0]

cohort_summary_clean = cohort_summary_clean[
    ["group_name", "index_name", "band_name", "value"]
]

In [ ]:
cohort_summary_clean.to_csv(OUTPUT_DIRS["tables"] / "ks_rehab_cohort_summary.csv", index=False)

In [ ]:
export_tasks = export_spatial_products_to_drive(
    spatial_products=spatial_products,
    aoi=merged_focal_geom,
    folder=SPATIAL_CHANGE_DRIVE_FOLDER,
    scale=S2_SCALE,
    crs=DEFAULT_CRS,
    prefix="s2",
)